In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
ccbp = pd.read_csv("data/Colorado_County_Business_Patterns_by_Industry__for_2000-2021.csv")

In [ ]:
ccbp.head()

## NAICS

### NAICS 2022

In [20]:
df = pd.read_excel("data/2-6 digit_2022_Codes.xlsx",engine="openpyxl",header=0)
df.columns

Index(['Seq. No.', '2022 NAICS US   Code', '2022 NAICS US Title', 'Unnamed: 3',
       'Unnamed: 4'],
      dtype='object')

In [ ]:
df['2022 NAICS US   Code']=  df['2022 NAICS US   Code'].astype(str)
df.set_index('2022 NAICS US   Code',inplace=True)
naics2022 = df['2022 NAICS US Title'].to_dict()
#print(naics2022)

In [89]:
ky = naics2022.copy().keys()

for naics in ky:
   if naics.find("-") > -1:
    nums = naics.split("-")
    dsc=naics2022[naics]
    for nn in range(int(nums[0]),int(nums[1])+1):
        naics2022[str(nn)] = dsc

### NAICS 2017

In [23]:
df = pd.read_excel("data/2-6 digit_2017_Codes.xlsx",engine="openpyxl",header=0)


In [24]:
df.columns

Index(['Seq. No.', '2017 NAICS US   Code', '2017 NAICS US Title', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5'],
      dtype='object')

In [25]:
df['2017 NAICS US   Code']=  df['2017 NAICS US   Code'].astype(str)
df.set_index('2017 NAICS US   Code',inplace=True)
naics2017 = df['2017 NAICS US Title'].to_dict()

In [90]:
ky = naics2017.copy().keys()

for naics in ky:
   if naics.find("-") > -1:
    nums = naics.split("-")
    dsc=naics2017[naics]
    for nn in range(int(nums[0]),int(nums[1])+1):
        naics2017[str(nn)] = dsc

### NAICS2012

In [26]:
df = pd.read_excel("data/2-digit_2012_Codes.xls",header=0)

In [27]:
df['2012 NAICS US   Code']=  df['2012 NAICS US   Code'].astype(str)

df.set_index('2012 NAICS US   Code',inplace=True)
naics2012 = df['2012 NAICS US Title'].to_dict()

In [80]:
ky = naics2012.copy().keys()

for naics in ky:
   if naics.find("-") > -1:
    nums = naics.split("-")
    dsc=naics2012[naics]
    for nn in range(int(nums[0]),int(nums[1])+1):
        naics2012[str(nn)] = dsc

In [ ]:
naics2012.keys()

### NAICS 2007

In [85]:
with open("data/naics07.txt","r") as fin:
    lines = fin.readlines()
    naics2007 = {}
    for line in lines[1:]:
        if (len(line) > 10):
           naics = line[8:14].strip()
           desc = line[16:].strip()
           if naics.find("-") > -1:
              nums = naics.split("-")
              for nn in range(int(nums[0]),int(nums[1])+1):
                naics2007[str(nn)] = dsc
           else:
              naics2007[naics] = desc
          

### NAICS 2002

In [71]:
with open("data/naics_2_6_02.txt","r") as fin:
    lines = fin.readlines()
    naics2002 = {}
    for line in lines[8:]:
        if (len(line) > 10):
           naics = line[:6].strip()
           desc = line[8:].strip()
           if naics.find("-") > -1:
              nums = naics.split("-")
              for nn in range(int(nums[0]),int(nums[1])+1):
                naics2002[str(nn)] = dsc
           else:
              naics2002[naics] = desc

In [ ]:
naics2002

### NAICS 1997

In [65]:
sects = {'11':'Agriculture, Forestry, Fishing and Hunting',
'21':'Mining',
'22':'Utilities',
'23':'Construction',
'31-33':'Manufacturing',
'42':'Wholesale Trade',
'44-45':'Retail Trade',
'48-49':'Transportation and Warehousing',
'51':'Information',
'52':'Finance and Insurance',
'53':'Real Estate and Rental and Leasing',
'54':'Professional, Scientific, and Technical Services',
'55':'Management of Companies and Enterprises',
'56':'Administrative and Support and Waste Management and Remediation Services',
'61':'Educational Services',
'62':'Health Care and Social Assistance',
'71':'Arts, Entertainment, and Recreation',
'72':'Accommodation and Food Services',
'81':'Other Services (except Public Administration)',
'92':'Public Administration'}

In [ ]:

naics1997={}
#sects = ["11","21","22","23","31-33","42","44-45","48-49","51","52","53","54","55","56","61","62","71","72","81","92"]
for num,dsc in sects.items():
    if num.find("-") > -1:
        nums = num.split("-")
        for nn in range(int(nums[0]),int(nums[1])+1):
            naics1997[str(nn)] = dsc
    else:
        naics1997[num]=dsc.strip()
    URL = f"https://www.census.gov/naics/resources/archives/sect{num}.html"
    page = requests.get(URL).text
    soup = BeautifulSoup(page, 'html.parser')
    print(num)
    for p in soup.find_all("h3"):
        s=p.text
        brk = s.find(" ")
        id = s[0:brk]
        desc = s[brk+1:].strip()
        naics1997[id]=desc
    
print(naics1997)



In [ ]:
naics1997.keys()

In [ ]:
ccbpN = list(ccbp["naics"].unique())

In [ ]:
cols = list(naics.columns)
cols

In [ ]:

names = {}

for nc in ccbpN:
    nco=nc
    if nc == "------":
       nc = "ALL Industries"
    else:
       nc = nc.replace("/","")
       nc = nc.replace("-","")    
#       print(nc,len(nc))
       try: 
          names[nc] = naics.loc[naics["naics22"].str[0:len(nc)] == nc,cols[len(nc)]].to_list()[0]
       except:
          print("Bad ",nco,nc,len(nc))
#       print("#####")

In [ ]:
naics["naics22"] = naics["naics22"].astype(str)

In [ ]:
naics["naics22"].str[0:4]

In [96]:
def mapIndustry(row):
    nc = row["naics"]
    year = row["year"]
    name=""
    if nc == "------":
       nc = "ALL"
       name="All Industries"    
    else:
        nc = nc.replace("/","")
        nc = nc.replace("-","")   
        if year  < 2003:
           if nc in naics1997:
              name=naics1997[nc]
           elif nc in naics2022:
              name=naics2022[nc]    
        if year  < 2008:
           if nc in naics2002:
              name=naics2002[nc]
           elif nc in naics2022:
              name=naics2022[nc]

        elif year  < 2012:
           if nc in naics2007:
              name=naics2007[nc]
           elif nc in naics2022:
              name=naics2022[nc]
        elif year < 2017:
           if nc in naics2012:
              name=naics2012[nc]
           elif nc in naics2022:
              name=naics2022[nc]
        elif year < 2022:
           if nc in naics2017:
              name=naics2017[nc]
           elif nc in naics2022:
              name=naics2022[nc]
        elif year <= 2023:
           if nc in naics2022:
              name=naics2022[nc]
    if ( nc == "95"):
        name="Administration Of Environmental Quality And Housing"
    elif (nc == "99"):
         name="UnClassifiable"
    if len(name) < 2:
        print(year,nc,name)
    return name
    
ccbp["industryName"] = ccbp.apply(mapIndustry,axis=1)

In [97]:
ccbp.columns

Index(['year', 'fipState', 'fipsCty', 'naics', 'empFlag', 'emp', 'qp1', 'ap',
       'est', 'n5_9', 'n10_19', 'n20_49', 'n50_99', 'n100_249', 'n250_499',
       'n500_999', 'n1000', 'n1000_1', 'n1000_2', 'n1000_3', 'n1000_4',
       'cenState', 'cenCty', 'emp_nf', 'qp1_nf', 'ap_nf', 'n<5',
       'industryName'],
      dtype='object')

In [98]:
ccbp.head()

,year,fipState,fipsCty,naics,empFlag,emp,qp1,ap,est,n5_9,...,n1000_2,n1000_3,n1000_4,cenState,cenCty,emp_nf,qp1_nf,ap_nf,n<5,industryName
0,2000,8,1,------,NaN,127505,966641,4093491,7221,1326,...,1,0,1,84.0,1.0,NaN,NaN,NaN,3705,All Industries
1,2000,8,1,11----,NaN,78,537,1991,9,2,...,0,0,0,84.0,1.0,NaN,NaN,NaN,5,"Agriculture, Forestry, Fishing and Hunting"
2,2000,8,1,113///,B,0,0,0,1,0,...,0,0,0,84.0,1.0,NaN,NaN,NaN,0,Forestry and Logging
3,2000,8,1,1131//,B,0,0,0,1,0,...,0,0,0,84.0,1.0,NaN,NaN,NaN,0,Timber Tract Operations
4,2000,8,1,11311/,B,0,0,0,1,0,...,0,0,0,84.0,1.0,NaN,NaN,NaN,0,Timber Tract Operations


In [105]:
ccbp["industryName"] = ccbp["industryName"].str.replace("\"","")

In [109]:
ccbp.loc[ccbp["industryName"].str.contains("\""),"industryName"]

Series([], Name: industryName, dtype: object)

In [110]:
ccbp.to_csv("data/Colorado_County_Business_Patterns_by_Industry__for_2000-2021_NAMES.csv")